In [1]:
# Environment
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found")

print("Environment configured successfully.")

Environment configured successfully.


In [2]:
# Import LangChain components
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

In [3]:
# Connect to our existing ChromaDB
embedding_model = OpenAIEmbeddings(
    api_key=os.environ["OPENAI_API_KEY"],
    model="text-embedding-3-small"
)

vector_store = Chroma(
    collection_name="CSV_RAG_BASELINE",
    persist_directory="../chroma_db",
    embedding_function=embedding_model
)

print("Connected to ChromaDB.")

Connected to ChromaDB.


In [4]:
# Verify our vector store
print(
    f"Vectors available: "
    f"{vector_store._collection.count()}"
)

Vectors available: 1538


In [5]:
# Create the Retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created.")

Retriever created.


In [7]:
# Test the Retriever
question = "Which product is associated with Renal Cell Carcinoma?"

retrieved_docs = retriever.invoke(question)

print(f"Retrieved {len(retrieved_docs)} documents.")

Retrieved 3 documents.


In [8]:
# Inspect retrieved documents
for i, doc in enumerate(retrieved_docs, start=1):

    print("=" * 80)
    print(f"Document {i}")
    print("Metadata:", doc.metadata)
    print("Content:")
    print(doc.page_content[:500])

Document 1
Metadata: {'source': '..\\data\\Pharma_Sales_Long.csv', 'row': 85}
Content:
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,
Document 2
Metadata: {'row': 135, 'source': '..\\data\\Pharma_Sales_Long.csv'}
Content:
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, C

In [9]:
# Convert Documents into Context
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

print("Context created.")
print("=" * 80)
print(context)

Context created.
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,

Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, custome

In [10]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the user's question using ONLY the context provided below.

If the answer cannot be found in the context, say:

"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
""")

print("Prompt created.")

Prompt created.


In [11]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("Chat model initialized.")

Chat model initialized.


In [12]:
# Traditional RAG — Before LCEL
formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(formatted_prompt)

messages=[HumanMessage(content='\nYou are a helpful AI assistant.\n\nAnswer the user\'s question using ONLY the context provided below.\n\nIf the answer cannot be found in the context, say:\n\n"I don\'t know based on the provided context."\n\nContext:\nNotes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,\n\nNotes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer q

In [13]:
response = llm.invoke(formatted_prompt)

print(response.content)

WELIREG is associated with Renal Cell Carcinoma.


In [ ]:
# Basic LCEL
# LCEL allows us to express the sequence of operations declaratively using the pipe operator.
chain = prompt | llm

print("LCEL chain created.")

LCEL chain created.


In [15]:
# Invoke the LCEL chain
response_lcel = chain.invoke({
    "context": context,
    "question": question
})

print(response_lcel.content)

WELIREG is associated with Renal Cell Carcinoma.


In [17]:
question = "What is the population of Japan?"

In [18]:
retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response.content)

I don't know based on the provided context.


## V1 Result

The end-to-end RAG pipeline successfully retrieved relevant information from the CSV dataset and generated a grounded response.

Question:
"What product is associated with Renal Cell Carcinoma?"

Retrieved information:
WELIREG records

Generated Answer:
"WELIREG is associated with Renal Cell Carcinoma."

The same prompt and LLM were also successfully executed using basic LCEL:

```python
chain = prompt | llm

In [19]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [20]:
retrieved_docs = retriever.invoke(
    "Which product is associated with Renal Cell Carcinoma?"
)

context = format_docs(retrieved_docs)

print(context)

Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,

Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,

Note

In [21]:
rag_chain = (
    {
        "context": lambda question: format_docs(
            retriever.invoke(question)
        ),
        "question": lambda question: question
    }
    | prompt
    | llm
)

In [22]:
question = "Which product is associated with Renal Cell Carcinoma?"

response = rag_chain.invoke(question)

print(response.content)

WELIREG is associated with Renal Cell Carcinoma.


In [23]:
question = "What is GARDASIL 9 associated with?"

response = rag_chain.invoke(question)

print(response.content)

GARDASIL 9 is associated with HPV prevention.


In [24]:
question = "What is Keytruda?"

response = rag_chain.invoke(question)

print(response.content)

KEYTRUDA is a product used in the treatment of Melanoma.
